# 🎬 AutoCut Video AI — Server Komputasi Cloud Colab (V1.4.0 Protected)

> ⏰ **Rilis:** V1.4.0 (Batch Recipe Serverless Edition)
> 🛡️ **Anti-Timeout Cloudflare:** Asynchronous Job Queue (`/api/v1/render/batch-recipe`)
> ⚡ **Ultra-Fast RAM-Disk:** Throughput 4.000 MB/s di `/dev/shm` (Bebas Bottleneck SSD)
> 🚀 **Zero Pop-Up Google Drive:** Server berjalan instan tanpa izin akses Drive (100% aman).
> 🔐 **Protected Core Engine:** Modul terenkripsi resmi PyArmor Linux x86_64.

Notebook ini berfungsi sebagai **Engine Backend Rendering Video Klip** untuk aplikasi **Intisari AutoCut Android**.

### ✨ Fitur Unggulan V1.4.0:
1. 📦 **Batch Recipe Serverless**: Unduh master video YouTube 1x, proses otomatis puluhan klip resep sekaligus.
2. 🎙️ **Sub-Second Groq Whisper LPU**: Transkripsi audio kilat <0.8s per klip tanpa beban memori GPU lokal.
3. 📱 **Scan-to-Pairing (QR Code)**: URL tunnel Cloudflare otomatis dikonversi ke gambar QR Code di layar Colab untuk pairing kamera 1-detik dari smartphone.
4. 🖥️ **MediaPipe Dynamic Face Tracking**: Pelacakan wajah otomatis berbasis AI untuk reframe vertikal 9:16.
5. 🎨 **6 Preset Visual Hook Card**: Kartu hook otomatis (Breaking News, TikTok Card, Neon, dll.) dengan Pillow RGBA.
6. ⏱️ **Live Countdown Watchdog**: Widget hitung mundur realtime di konsol Colab & auto-shutdown runtime saat idle untuk menghemat kuota GPU.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Tunggu hingga proses setup selesai dan **Tautan Tunnel**, **Spesifikasi Server**, serta **QR Code** muncul di layar.
3. Buka aplikasi **Intisari AutoCut Android** di HP Anda -> Buka Tab **Pengaturan** -> Pindai QR Code atau salin tautan tersebut.


In [ ]:
"""
🎬 AUTOCUT VIDEO ENGINE — SERVER RENDERING & CLOUDFLARE TUNNEL (V1.4.0)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title ⚙️ PUSAT KENDALI ENGINE RENDERING COLAB
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 10  # @param [0, 5, 10, 15, 30] {type:"raw"}
GROQ_API_KEY = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import re
import json
import shutil
import zipfile
import subprocess
import sysconfig
import threading
import urllib.request

print("=" * 80)
print("🚀 MEMULAI AUTOCUT VIDEO ENGINE V1.4.0 (BYOC SERVER)")
print("=" * 80)

# 1. Download binary cloudflared jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ [1/5] Mengunduh Cloudflare Tunnel client...")
    try:
        subprocess.run([
            "wget", "-q", "-O", "/usr/local/bin/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        ], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
        print("✅ Cloudflare Tunnel siap!")
    except Exception as e_cf:
        print(f"⚠️ Gagal memasang cloudflared: {e_cf}")
else:
    print("✅ Cloudflare Tunnel sudah terpasang.")

# 2. Install dependensi sistem dasar & fast libraries
print("📦 [2/5] Memeriksa dependensi sistem (FastAPI, yt-dlp terbaru, mediapipe, qrcode)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn[standard]", "python-multipart", "mediapipe", "requests", "qrcode", "pydantic"
], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp"], check=False)

# Deteksi dan sinkronisasi Pillow jika terjadi ketidakcocokan C-extension / _Ink
try:
    from PIL import Image, ImageDraw
except Exception as e_pil_chk:
    print(f"\n⚡ [AUTO-RESTART] Menyelaraskan pustaka Pillow di Colab ({e_pil_chk})...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "pillow"], check=False)
    time.sleep(1.0)
    os.kill(os.getpid(), 9)

# 3. Unduh modul engine biner resmi terenkripsi (Multi-Source Fallback Downloader)
print("🔐 [3/5] Mengunduh modul biner terenkripsi: autocut_video_engine.zip...")
pkg_candidates = [
    f"https://raw.githubusercontent.com/intisariapps-com/Intisari-AutoCut-Android/main/autocut_video_engine.zip?t={int(time.time())}",
    f"https://raw.githubusercontent.com/intisariapps-com/Intisari-AutoCut-Android/dev/autocut_video_engine.zip?t={int(time.time())}",
    "https://cdn.jsdelivr.net/gh/intisariapps-com/Intisari-AutoCut-Android@main/autocut_video_engine.zip"
]
pkg_local = "/tmp/autocut_video_engine.zip"
download_success = False

for candidate_url in pkg_candidates:
    try:
        req = urllib.request.Request(candidate_url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
        with urllib.request.urlopen(req, timeout=15) as response, open(pkg_local, "wb") as out_file:
            shutil.copyfileobj(response, out_file)
        if os.path.exists(pkg_local) and zipfile.is_zipfile(pkg_local) and os.path.getsize(pkg_local) > 50000:
            download_success = True
            pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1)
            print(f"✅ Paket Biner Berhasil Diunduh ({pkg_size_kb} KB) dari sumber terpercaya.")
            break
    except Exception as e_dl:
        continue

if not download_success:
    print("⚠️ Mencoba pengunduhan via wget fallback...")
    for candidate_url in pkg_candidates:
        try:
            subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, candidate_url], check=True)
            if os.path.exists(pkg_local) and zipfile.is_zipfile(pkg_local):
                download_success = True
                break
        except Exception:
            continue

if not download_success or not zipfile.is_zipfile(pkg_local):
    raise RuntimeError("❌ GAGAL MENGUNDUH BINER ENGINE! Pastikan repositori Intisari-AutoCut-Android dapat diakses.")

site_pkg = sysconfig.get_paths()["purelib"]
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Bersihkan cache modul Python agar selalu memuat biner terbaru
for mod in list(sys.modules.keys()):
    if "autocut_video_engine" in mod or "pyarmor" in mod or "PIL" in mod:
        del sys.modules[mod]

import autocut_video_engine
eng_ver = getattr(autocut_video_engine, "__version__", "1.4.1")
print(f"✅ Modul Engine AutoCut (Versi: v{eng_ver}) berhasil dipasang di memori!")

# 4. Setel environment & Hugging Face token
if GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
    print("✅ Hugging Face Terautentikasi (Kuota Unduh Tinggi Aktif!)")

# 5. Unduh & Muat Model AI Whisper ke GPU VRAM di awal (Zero-Delay Warm-Up)
print("⚡ [4/5] Menginisialisasi AI Engine (Groq Whisper Cloud LPU & MediaPipe Face Tracking)...")
print("🌐 [5/5] Meluncurkan server backend pada port 8000...")
import uvicorn
from autocut_video_engine.server import app as fastapi_app

def start_uvicorn():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2.5)

# 5. Jalankan Cloudflare Tunnel dan ambil URL publik
print("🚇 Membuka Cloudflare Quick Tunnel...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    universal_newlines=True
)

public_tunnel_url = None
timeout_sec = 25
start_t = time.time()
while time.time() - start_t < timeout_sec:
    line = tunnel_proc.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_tunnel_url = match.group(0)
            break
    time.sleep(0.1)

if not public_tunnel_url:
    print("⚠️ Gagal mendapatkan URL Cloudflare otomatis. Periksa koneksi.")
else:
    # Ambil telemetri hardware server dari health endpoint lokal
    telemetry_data = {}
    try:
        import urllib.request
        with urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health", timeout=3) as h_res:
            h_json = json.loads(h_res.read().decode("utf-8"))
            telemetry_data = h_json.get("telemetry", {})
    except Exception:
        pass

    gpu_name = telemetry_data.get("gpu", {}).get("name", "Tidak Terdeteksi")
    gpu_vram = telemetry_data.get("gpu", {}).get("vram_total_mb", 0)
    cpu_model = telemetry_data.get("cpu", {}).get("model", "Multi-Core")
    cpu_cores = telemetry_data.get("cpu", {}).get("cores", 2)
    ram_total = telemetry_data.get("memory", {}).get("ram_total_mb", 0)
    ram_disk_mb = telemetry_data.get("ram_disk", {}).get("total_mb", 0)
    encoder = telemetry_data.get("capabilities", {}).get("video_encoder", "libx264")
    benchmark = telemetry_data.get("capabilities", {}).get("benchmark_estimate", "45-60s/klip")

    print("\n" + "=" * 80)
    print("🎉 SERVER AUTOCUT VIDEO ENGINE BERHASIL ONLINE!")
    print("=" * 80)
    print(f"👉 URL TUNNEL PUBLIK : {public_tunnel_url}")
    print("-" * 80)
    print("🖥️  PROFIL SPESIFIKASI SERVER COLAB:")
    print(f"   • GPU Hardware       : {gpu_name} ({gpu_vram} MB VRAM)")
    print(f"   • CPU Processor      : {cpu_model} ({cpu_cores} Cores)")
    print(f"   • RAM Sistem         : {ram_total} MB")
    print(f"   • RAM-Disk (/dev/shm): {ram_disk_mb} MB (Buffer Cepat 4.000 MB/s)")
    print(f"   • Encoder Video      : {encoder.upper()} (Akselerasi Aktif)")
    print(f"   • Estimasi Kecepatan : {benchmark}")
    print("=" * 80)

    # Tampilkan QR Code di layar Colab untuk pairing kamera smartphone
    try:
        import qrcode
        from IPython.display import display, Image
        import io
        qr = qrcode.QRCode(box_size=7, border=2)
        qr.add_data(public_tunnel_url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        print("\n📱 PINDAI QR CODE DI BAWAH INI DARI APLIKASI ANDROID:")
        display(Image(buf.getvalue()))
    except Exception as e_qr:
        print(f"(QR Code viewer fallback: {e_qr})")

    print("\nℹ️ Server siap menerima instruksi render video dari aplikasi Android!")
    print("⏱️ Tekan tombol Stop (⏹) kapan saja untuk mematikan server.")

# 6. Live Interactive Watchdog Timer (ATM dari intiVoice V1.6.3)
def _idle_watchdog_loop():
    try:
        from autocut_video_engine.server import watchdog
    except ImportError:
        try:
            from autocut_video_engine.watchdog import watchdog
        except ImportError:
            from colab.engine.watchdog import watchdog
    timeout_min = float(AUTO_SHUTDOWN_MINUTES)
    if timeout_min <= 0:
        print("⏱️ [AUTO-SHUTDOWN NONAKTIF] Mesin akan standby tanpa batas waktu.")
        while True:
            time.sleep(1)
        return

    idle_limit_sec = timeout_min * 60.0
    display_handle = None
    try:
        from IPython.display import display, HTML
        init_html = (
            "<div style='font-family: monospace; font-size: 13px; color: #00D2B4; background: #071952; "
            "padding: 10px 16px; border-radius: 10px; border: 1px solid #1A73E8; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
            "⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Menyiapkan timer realtime..."
            "</div>"
        )
        display_handle = display(HTML(init_html), display_id=True)
    except Exception:
        display_handle = None

    while True:
        time.sleep(1.0)
        try:
            is_active = getattr(watchdog, "is_busy", getattr(watchdog, "active_jobs", 0) > 0)
            if is_active:
                watchdog.touch()

            elapsed_idle = time.time() - watchdog.last_activity
            remaining_sec = max(0, int(idle_limit_sec - elapsed_idle))
            mins = remaining_sec // 60
            secs = remaining_sec % 60

            if display_handle:
                try:
                    if is_active:
                        widget_html = (
                            "<div style='font-family: monospace; font-size: 13px; color: #FFD700; background: #1a1a00; "
                            "padding: 10px 16px; border-radius: 10px; border: 1px solid #FFA500; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            "⚡ <b>ENGINE SEDANG MERENDER:</b> Memproses video klip... <span style='color: #00D2B4;'>(Timer idle dijeda)</span>"
                            "</div>"
                        )
                    else:
                        badge_color = "#00D2B4" if remaining_sec > 60 else ("#FFA500" if remaining_sec > 30 else "#FF4444")
                        bg_color = "#071952" if remaining_sec > 60 else ("#2b1700" if remaining_sec > 30 else "#2b0000")
                        border_color = "#1A73E8" if remaining_sec > 60 else ("#FF8C00" if remaining_sec > 30 else "#FF0000")

                        widget_html = (
                            f"<div style='font-family: monospace; font-size: 13px; color: {badge_color}; background: {bg_color}; "
                            f"padding: 10px 16px; border-radius: 10px; border: 1px solid {border_color}; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            f"⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Sisa Waktu Idle: <b style='font-size: 16px; color: #FFFFFF;'>{mins:02d}:{secs:02d}</b> "
                            f"<span style='font-size: 11px; opacity: 0.8;'>| Reset otomatis tiap ada job render baru</span>"
                            f"</div>"
                        )
                    from IPython.display import HTML
                    display_handle.update(HTML(widget_html))
                except Exception:
                    pass

            if elapsed_idle >= idle_limit_sec and not is_active:
                print(f"\n🛑 [AUTO-SHUTDOWN] TIDAK ADA AKTIVITAS RENDER SELAMA {int(timeout_min)} MENIT.")
                print("💡 Memutuskan runtime Google Colab untuk menghemat kuota compute units...")
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception:
                    os._exit(0)
                return
        except Exception:
            pass

try:
    _idle_watchdog_loop()
except KeyboardInterrupt:
    print("\n🛑 Sesi server dihentikan oleh pengguna.")
